# **MÓDULO 35 - Cross Validation**

Nesta tarefa, você trabalhará com uma base de dados que contém informações sobre variáveis ambientais coletadas para a detecção de incêndios. O objetivo é utilizar técnicas de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação na previsão da ocorrência de um incêndio com base nas variáveis fornecidas.


Descrição da Base de Dados
A base de dados contém as seguintes variáveis:

Unnamed:0: Índice (não é uma variável útil para o modelo)

UTC: Tempo em Segundos UTC

Temperature[C]: Temperatura do Ar (em graus Celsius)

Humidity[%]: Umidade do Ar (em porcentagem)

TVOC[ppb]: Total de Compostos Orgânicos Voláteis (medido em partes por bilhão)

eCO2[ppm]: Concentração equivalente de CO2 (medido em partes por milhão)

Raw H2: Hidrogênio molecular bruto, não compensado

Raw Ethanol: Etanol gasoso bruto

Pressure[hPA]: Pressão do Ar (em hectopascais)

PM1.0: Material particulado de tamanho < 1,0 µm

PM2.5: Material particulado de tamanho >1,0 µm e < 2,5 µm

NC0.5: Concentração numérica de material particulado de tamanho < 0,5 µm

NC1.0: Concentração numérica de material particulado de tamanho 0,5 µm < 1,0 µm

NC2.5: Concentração numérica de material particulado de tamanho 1,0 µm < 2,5 µm

CNT: Contador de amostras


E a variável alvo:

Fire Alarm: Indicador binário de incêndio (1 se houver incêndio, 0 caso contrário)

O objetivo desta tarefa é aplicar a técnica de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação. A validação cruzada ajudará a garantir que o modelo seja avaliado de maneira robusta e generalize bem para dados não vistos.

In [8]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

# 1 - Carregue a base de dados, verifique os tipos de dados e também se há presença de dados faltantes ou nulos.

In [11]:
df = pd.read_csv("smoke_detection_iot.csv")

print(df.shape)
print(df.head())
print(df.info())
print("\nNulos:")
print(df.isnull().sum())

# Renomear coluna
df.rename(columns={'Fire Alarm': 'Fire_Alarm'}, inplace=True)

(62630, 16)
   Unnamed: 0         UTC  Temperature[C]  Humidity[%]  TVOC[ppb]  eCO2[ppm]  \
0           0  1654733331          20.000        57.36          0        400   
1           1  1654733332          20.015        56.67          0        400   
2           2  1654733333          20.029        55.96          0        400   
3           3  1654733334          20.044        55.28          0        400   
4           4  1654733335          20.059        54.69          0        400   

   Raw H2  Raw Ethanol  Pressure[hPa]  PM1.0  PM2.5  NC0.5  NC1.0  NC2.5  CNT  \
0   12306        18520        939.735    0.0    0.0    0.0    0.0    0.0    0   
1   12345        18651        939.744    0.0    0.0    0.0    0.0    0.0    1   
2   12374        18764        939.738    0.0    0.0    0.0    0.0    0.0    2   
3   12390        18849        939.736    0.0    0.0    0.0    0.0    0.0    3   
4   12403        18921        939.744    0.0    0.0    0.0    0.0    0.0    4   

   Fire Alarm  
0   

Para a coluna Fire Alarm, por conta do espaçamento talvez seja util renomear o nome da coluna utilizando:

df.rename(columns={'Fire Alarm': 'Fire_Alarm'}, inplace=True)

### Etapa 1 - Verificacao dos Dados

A base possui 62.630 linhas e 16 colunas, todas numericas
(int64 ou float64). Nenhum valor nulo foi encontrado em
nenhuma coluna. A unica acao necessaria e remover a coluna
Unnamed: 0, que e apenas um indice sem valor analitico,
e renomear Fire Alarm para Fire_Alarm por causa do espaco
no nome.

# 2 - Para essa base, onde você realizará as previsões de fire alarm, qual modelo de machine learning você aplicará? Justifique.




### Etapa 2 - Escolha do Modelo

O modelo escolhido foi a Regressao Logistica. A justificativa
e que o problema e de classificacao binaria (Fire_Alarm: 0 ou 1),
exatamente o tipo de problema para o qual a Regressao Logistica
foi desenhada. Alem disso, e um modelo simples, rapido de
treinar mesmo com 62 mil registros, e interpretavel atraves
dos coeficientes, permitindo entender quais variaveis
ambientais mais influenciam a deteccao de incendio.

# 3 - Separe a base em Y e X e já rode a instância do modelo que você utilizará.

In [15]:
# Remover coluna indice
df = df.drop(columns=['Unnamed: 0'])

X = df.drop(columns=['Fire_Alarm'])
y = df['Fire_Alarm']

modelo = LogisticRegression(max_iter=1000, random_state=42)

print("X:", X.shape)
print("y:", y.shape)
print("\nBalanceamento:")
print(y.value_counts(normalize=True).mul(100).round(1))

X: (62630, 14)
y: (62630,)

Balanceamento:
Fire_Alarm
1    71.5
0    28.5
Name: proportion, dtype: float64


# 4 - Defina o número de Folds e rode o modelo com a validação cruzada.

In [17]:
# Definir o KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Rodar a validacao cruzada
scores = cross_val_score(modelo, X, y, cv=kf, scoring='accuracy')

print("Scores por fold:")
print(scores)

Scores por fold:
[0.97668849 0.98626856 0.97844483 0.97708766 0.9776465 ]


# 5 - Avalie a pontuação de cada modelo e ao final a validação final da média.

In [19]:
print("Scores individuais por fold:")
for i, score in enumerate(scores, 1):
    print(f"Fold {i}: {score:.4f}")

print(f"\nMedia de acuracia: {scores.mean():.4f}")
print(f"Desvio padrao: {scores.std():.4f}")

Scores individuais por fold:
Fold 1: 0.9767
Fold 2: 0.9863
Fold 3: 0.9784
Fold 4: 0.9771
Fold 5: 0.9776

Media de acuracia: 0.9792
Desvio padrao: 0.0036


### Etapa 5 - Avaliacao Final

Scores por fold:
- Fold 1: 0.9767
- Fold 2: 0.9863
- Fold 3: 0.9784
- Fold 4: 0.9771
- Fold 5: 0.9776

Media de acuracia: 0.9792 (97.92%)
Desvio padrao: 0.0036

O modelo apresentou desempenho excelente e extremamente
consistente entre os folds. O desvio padrao muito baixo
(0.0036) indica que o modelo generaliza bem e nao depende
de uma divisao especifica dos dados para funcionar bem.

Isso sugere que as variaveis ambientais (temperatura,
umidade, particulas, gases) tem uma relacao muito clara
e forte com a ocorrencia de incendio, permitindo que mesmo
um modelo simples como a Regressao Logistica alcance
quase 98% de acuracia de forma robusta.